In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import gc
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
data_train_dir2='train_2'
data_test_dir2='val_2'

In [ ]:
print(os.listdir(data_train_dir2))
num_classes=len(os.listdir(data_train_dir2))

print(os.listdir(data_test_dir2))
num_classes2=len(os.listdir(data_test_dir2))

Bar Chart

In [ ]:
import os
import matplotlib.pyplot as plt


train_dir = 'train_2'  

# Classes
categories = ['0','1']

#Empty List
image_counts = []

# Loop through each category 
for category in categories:
    category_path = os.path.join(train_dir, category)
    image_count = len([f for f in os.listdir(category_path) if os.path.isfile(os.path.join(category_path, f))])
    image_counts.append(image_count)

# Bar Plot
plt.figure(figsize=(10, 6))
plt.bar(categories, image_counts, color='skyblue')
plt.xlabel('Alzheimer Categories')
plt.ylabel('Number of Images')
plt.title('Distribution of Alzheimer Categories in Dataset')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt


train_dir = 'val_2'  

# Classes
categories = ['0','1']

# Empty List
image_counts = []

# Loop through each category 
for category in categories:
    category_path = os.path.join(train_dir, category)
    image_count = len([f for f in os.listdir(category_path) if os.path.isfile(os.path.join(category_path, f))])
    image_counts.append(image_count)

# Bar Plot
plt.figure(figsize=(10, 6))
plt.bar(categories, image_counts, color='skyblue')
plt.xlabel('Alzheimer Categories')
plt.ylabel('Number of Images')
plt.title('Distribution of Alzheimer Categories in Dataset')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Offline Augmentation

In [ ]:
import os
import random
from PIL import Image
from collections import Counter

import torch
from torch.utils.data import Dataset, ConcatDataset, DataLoader
from torchvision import datasets, transforms

# --- Configuration ---
dataset_root = 'train_2'          
minority_class = '0'                
majority_class = '1'                
save_augmented_to_disk = False      
aug_save_dir = os.path.join(dataset_root, minority_class + '_augmented')
target_count = 24384                

# --- Transforms ---
augmentation_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
])


def is_image_valid(path):
    try:
        img = Image.open(path)
        img.verify()
        return True
    except:
        return False

minority_dir = os.path.join(dataset_root, minority_class)
minority_image_paths = [
    os.path.join(minority_dir, fname)
    for fname in os.listdir(minority_dir)
    if is_image_valid(os.path.join(minority_dir, fname))
]


# Generate Augmented Images
augmented_images = []
needed = target_count - len(minority_image_paths)

if save_augmented_to_disk:
    os.makedirs(aug_save_dir, exist_ok=True)

for i in range(needed):
    img_path = random.choice(minority_image_paths)
    img = Image.open(img_path).convert('RGB')
    augmented_tensor = augmentation_transforms(img)
    
    if save_augmented_to_disk:
        to_pil = transforms.ToPILImage()
        aug_img_pil = to_pil(augmented_tensor)
        save_path = os.path.join(aug_save_dir, f'aug_{i}.jpg')
        aug_img_pil.save(save_path)
    
    augmented_images.append((augmented_tensor, 0))  # Label 0

# Load dataset
original_dataset = datasets.ImageFolder(dataset_root, transform=transforms.ToTensor())


# Wrap augmented data
class AugmentedDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

aug_dataset = AugmentedDataset(augmented_images)

# Combine the dataset
balanced_dataset = ConcatDataset([original_dataset, aug_dataset])
balanced_loader = DataLoader(balanced_dataset, batch_size=32, shuffle=True)
